# 03_finetune.ipynb — Fine-tuning Pipeline

**Project:** SentinelFatal2 — Foundation Model for Fetal Distress Prediction  
**Step:** 4 — Fine-tuning for Acidemia Classification  
**Source:** arXiv:2601.06149v1, Section II-E  
**SSOT:** `docs/work_plan.md`, Part ו שלב 4

---

## Notebook structure

| Cell | Purpose |
|------|----------|
| 1 | Setup: paths, seed, sys.path (+ Colab Drive mount) |
| 2 | GPU check |
| 3 | Load config |
| 4-11 | **V4.1 – V4.8** Validation checks |
| 12 | Summary |
| 13 | **Full fine-tuning** (or dry-run) |
| 14 | Verify artifacts |
| 15 | Plot AUC / loss curves |

**Critical constraints:**
- ❌ **NO test.csv access** (data leakage prevention — G4.5, G4.9, V4.5)
- ✅ Class weights from train.csv only (~[1.0, 3.9])
- ✅ Differential LR: backbone 1e-5, head 1e-4
- ✅ AUC per-recording with max aggregation (P7 fix)
- ✅ Gradient clipping max_norm=1.0
- ✅ Early stopping patience=15 on val AUC

## Cell 1 — Setup: paths, seed, Colab Drive mount

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os, random
import numpy as np
import torch

# ── Colab: mount Drive and cd to project ──────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/SentinelFatal2'   # adjust if needed
    if not os.path.isdir(PROJECT_ROOT):
        print('Project folder not found on Drive — cloning from GitHub...')
        os.system(f'git clone https://github.com/ArielShamay/SentinelFatal2.git "{PROJECT_ROOT}"')
    os.chdir(PROJECT_ROOT)
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
else:
    # Local / VS Code
    from pathlib import Path
    PROJECT_ROOT = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

print(f'Project root : {os.getcwd()}')
print(f'Python       : {sys.version}')
print(f'PyTorch      : {torch.__version__}')

## Cell 2 — GPU check

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    DEVICE = torch.device('cpu')
    print('No GPU found -- running on CPU (training will be slow)')

print(f'torch {torch.__version__}  |  device: {DEVICE}')

## Cell 3 — Load configuration + imports

In [ ]:
# ── Cell 3: Load config + imports ─────────────────────────────────────────────
from pathlib import Path
import torch.nn as nn

from src.model.patchtst import PatchTST, load_config
from src.model.heads import ClassificationHead
from src.data.dataset import FinetuneDataset, build_finetune_loaders
from src.train.utils import compute_recording_auc, sliding_windows
from src.train.finetune import compute_class_weights

CONFIG_PATH = Path(PROJECT_ROOT) / 'config' / 'train_config.yaml'
cfg = load_config(CONFIG_PATH)

print('Fine-tuning configuration:')
ftcfg = cfg['finetune']
for k, v in ftcfg.items():
    print(f'  {k}: {v}')

## V4.2 — Single Batch Training Test

**Requirement:** Training loop (forward -> loss -> backward -> step) runs on 1 batch without errors

In [ ]:
# Build model with pretrained backbone (if available) or random init
pr = Path(PROJECT_ROOT)
model = PatchTST(cfg)
pretrain_ckpt = pr / 'checkpoints' / 'pretrain' / 'best_pretrain.pt'
if pretrain_ckpt.exists():
    model.load_state_dict(torch.load(pretrain_ckpt, map_location=DEVICE, weights_only=True))
    print(f'Loaded pretrained checkpoint: {pretrain_ckpt}')
else:
    print('WARNING: No pretrained checkpoint found, using random init')

# Replace head
d_in = cfg['data']['n_patches'] * cfg['model']['d_model'] * cfg['data']['n_channels']
model.replace_head(ClassificationHead(d_in=d_in, n_classes=2, dropout=0.2))
model = model.to(DEVICE)

# Build data loader
train_loader, _ = build_finetune_loaders(
    train_csv=pr / 'data' / 'splits' / 'train.csv',
    val_csv=pr / 'data' / 'splits' / 'val.csv',
    processed_root=pr / 'data' / 'processed',
    batch_size=4,
)

# Setup optimizer and loss
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
backbone_params = list(model.patch_embed.parameters()) + list(model.encoder.parameters())
head_params = list(model.head.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 1e-4},
], weight_decay=1e-2)

# V4.2 Check: Single batch forward-backward-step
model.train()
batch_x, batch_y = next(iter(train_loader))
batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

optimizer.zero_grad()
logits = model(batch_x)
loss = criterion(logits, batch_y)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

# Validate
assert not torch.isnan(loss), 'Loss is NaN!'
assert not torch.isinf(loss), 'Loss is Inf!'
assert loss.item() > 0, 'Loss should be positive'

print(f'\n[V4.2] PASS - Single batch training successful')
print(f'  Batch shape: {batch_x.shape}, Labels: {batch_y.shape}')
print(f'  Logits shape: {logits.shape}')
print(f'  Loss: {loss.item():.6f}')

## V4.3 — AUC Function Sanity Check

**Requirement:** `compute_recording_auc()` returns value in [0, 1] on val set

In [ ]:
# V4.3 Check: Compute AUC on validation set (small stride for speed)
pr = Path(PROJECT_ROOT)
val_csv = pr / 'data' / 'splits' / 'val.csv'
processed_root = pr / 'data' / 'processed'

model.eval()
val_auc = compute_recording_auc(
    model, val_csv, processed_root, stride=60, device=str(DEVICE)
)

# Validate
assert 0.0 <= val_auc <= 1.0, f'AUC should be in [0, 1], got {val_auc}'

print(f'\n[V4.3] PASS - AUC computation successful')
print(f'  Val AUC: {val_auc:.6f}')
print(f'  (Using stride=60 for speed; official eval uses stride=1)')

## V4.4 — AUC Per-Recording Verification (Code Review)

**Requirement:** AUC is per-recording with max aggregation, NOT per-window  
**P7 Fix:** Training unit = window, Evaluation unit = recording

In [ ]:
import inspect

# V4.4 Check: Verify max aggregation in compute_recording_auc source
source = inspect.getsource(compute_recording_auc)

# Check for key patterns
assert 'max(scores)' in source, "Missing max aggregation!"
assert 'recording_score' in source, "Missing recording-level score!"
assert 'roc_auc_score' in source, "Missing ROC-AUC computation!"

# Print relevant code snippet
print("[V4.4] PASS - AUC computed per-recording with max aggregation")
print("\nKey code snippet:")
for line in source.split('\n'):
    if 'max(scores)' in line or 'recording_score' in line:
        print(f"  {line.strip()}")

## V4.5 — Test Data Verification

**Requirement:** Test set (test.csv) doesn't appear anywhere in code  
**Critical:** This is a data leakage check (G4.5, G4.9)

In [ ]:
# V4.5 Check: Grep for test.csv in production code (NOT validation notebooks)
pr = Path(PROJECT_ROOT)
files_to_check = [
    'src/data/dataset.py',
    'src/train/finetune.py',
    'src/train/utils.py',
]

violations = []
for fname in files_to_check:
    fpath = pr / fname
    if fpath.exists():
        content = fpath.read_text(encoding='utf-8')
        if 'test.csv' in content:
            violations.append(fname)

# Validate
assert len(violations) == 0, f'CRITICAL: test.csv found in: {violations}'

print('[V4.5] PASS - NO test.csv access detected in production code')
print(f'  Checked files: {len(files_to_check)}')
print(f"  Files: {', '.join(files_to_check)}")
print('  Data leakage prevention: OK')

## V4.6 — Differential Learning Rate Verification

**Requirement:** Differential LR: backbone=1e-5, head=1e-4

In [ ]:
# V4.6 Check: Inspect optimizer param groups
print("[V4.6] Optimizer parameter groups:")
for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group['params'])
    lr = group['lr']
    name = 'backbone' if i == 0 else 'head'
    print(f"  Group {i} ({name}): lr={lr:.2e}, params={n_params:,}")

# Validate
lr_backbone = optimizer.param_groups[0]['lr']
lr_head = optimizer.param_groups[1]['lr']

assert lr_backbone == 1e-5, f"Backbone LR should be 1e-5, got {lr_backbone}"
assert lr_head == 1e-4, f"Head LR should be 1e-4, got {lr_head}"
assert lr_head > lr_backbone, "Head LR should be higher than backbone LR"

print(f"\n[V4.6] PASS - Differential LR verified")
print(f"  Backbone LR: {lr_backbone:.2e} (prevents catastrophic forgetting)")
print(f"  Head LR:     {lr_head:.2e} (faster adaptation)")

## V4.7 — Gradient Clipping Verification

**Requirement:** Gradient clipping max_norm=1.0 active

In [ ]:
import inspect

# V4.7 Check: Verify gradient clipping in finetune.py source
from src.train import finetune
source = inspect.getsource(finetune.run_epoch)

# Check for gradient clipping call
assert 'clip_grad_norm_' in source, "Missing gradient clipping!"
assert 'max_norm' in source, "Missing max_norm parameter!"

# Check config value
clip_norm = cfg['finetune']['gradient_clip']
assert clip_norm == 1.0, f"gradient_clip should be 1.0, got {clip_norm}"

print("[V4.7] PASS - Gradient clipping verified")
print(f"  max_norm: {clip_norm}")
print("  Applied in: run_epoch() before optimizer.step()")

## V4.8 — Early Stopping Logic Check

**Requirement:** Early stopping on val AUC with patience=15

In [ ]:
import inspect

# V4.8 Check: Verify early stopping in finetune.py
source = inspect.getsource(finetune.train)

# Check for early stopping patterns
assert 'best_val_auc' in source, "Missing best_val_auc tracking!"
assert 'patience_ctr' in source or 'patience_counter' in source, "Missing patience counter!"
assert 'val_auc > best_val_auc' in source, "Missing AUC improvement check!"

# Check config value
patience = cfg['finetune']['patience']
assert patience == 15, f"patience should be 15, got {patience}"

print("[V4.8] PASS - Early stopping logic verified")
print(f"  Patience: {patience} epochs")
print("  Metric: validation AUC (per-recording)")
print("  Direction: maximize")

## Summary — All Validation Checks

If all cells above passed, Agent 4 implementation is complete and ready for handoff.

In [ ]:
print('=' * 60)
print('AGENT 4 VALIDATION SUMMARY')
print('=' * 60)
print('[V4.1] Class weights computed from train.csv: PASS')
print('[V4.2] Single batch training test: PASS')
print('[V4.3] AUC function returns [0,1]: PASS')
print('[V4.4] AUC per-recording (max aggregation): PASS')
print('[V4.5] NO test.csv access (data leakage check): PASS')
print('[V4.6] Differential LR (backbone=1e-5, head=1e-4): PASS')
print('[V4.7] Gradient clipping (max_norm=1.0): PASS')
print('[V4.8] Early stopping (patience=15, metric=val_auc): PASS')
print('=' * 60)
print('\nAll validation checks passed!')
print('Proceed to full training below.')

## Full Fine-tuning Run

Set `DRY_RUN = False` for full training on Colab GPU.  
Set `DRY_RUN = True` (default) for 2-batch CPU verification.

In [ ]:
# ── Full fine-tuning ──────────────────────────────────────────────────────────
#
# Set DRY_RUN = True (default) to run 2 batches only (CPU verification).
# Set DRY_RUN = False + DEVICE = cuda for the full Colab training run.
#
# NOTE: Calls train() directly (not via subprocess) so that:
#   1. Errors surface immediately with full traceback
#   2. Print output streams in real-time
#   3. Dataset loading issues are never hidden
#
from src.train.finetune import train as finetune_train

DRY_RUN    = True        # <-- change to False for full training on Colab
MAX_BATCHES = 2 if DRY_RUN else 0

if DRY_RUN:
    print(f'[dry-run] Running 2 batches on {DEVICE} ...')

finetune_train(
    config_path=str(CONFIG_PATH),
    device_str=str(DEVICE),
    max_batches=MAX_BATCHES,
)

if DRY_RUN:
    print('\nDry-run completed successfully')
else:
    print('\nFull fine-tuning complete')

In [ ]:
# ── Verify artifacts ──────────────────────────────────────────────────────────
import os

ckpt_path = 'checkpoints/finetune/epoch_000.pt'
best_path = 'checkpoints/finetune/best_finetune.pt'
assert os.path.exists(ckpt_path), f'FAIL: checkpoint not found at {ckpt_path}'
assert os.path.exists(best_path), f'FAIL: best checkpoint not found at {best_path}'

# Verify checkpoint loads correctly
loaded = torch.load(best_path, map_location='cpu', weights_only=True)
assert isinstance(loaded, dict), 'FAIL: checkpoint is not a state dict'
print(f'Checkpoints saved OK ({len(loaded)} param tensors)')

# Loss / AUC CSV logged
log_path = 'logs/finetune_loss.csv'
assert os.path.exists(log_path), f'FAIL: log not found at {log_path}'
import csv
with open(log_path) as f:
    rows = list(csv.DictReader(f))
assert len(rows) >= 1, 'FAIL: no rows in loss CSV'
print(f'Loss CSV has {len(rows)} row(s)')
for r in rows[:3]:
    print(f"  epoch={r.get('epoch','?')} train_loss={r.get('train_loss','?')} val_auc={r.get('val_auc','?')}")

In [ ]:
# ── Plot loss / AUC curves ────────────────────────────────────────────────────
import csv
import matplotlib.pyplot as plt

log_path = 'logs/finetune_loss.csv'
if not os.path.exists(log_path):
    print(f'Log file not found: {log_path} -- run training cell first.')
else:
    with open(log_path) as f:
        rows = list(csv.DictReader(f))

    if len(rows) < 2:
        print(f'Only {len(rows)} epoch(s) logged -- not enough for a meaningful plot.')
        print('Data:', rows)
    else:
        epochs     = [int(r['epoch'])        for r in rows]
        train_loss = [float(r['train_loss']) for r in rows]
        val_auc    = [float(r['val_auc'])    for r in rows]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(epochs, train_loss, marker='.', label='Train CE Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Cross-Entropy Loss')
        ax1.set_title('Fine-tuning Training Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, val_auc, marker='.', color='green', label='Val AUC')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('AUC')
        ax2.set_title('Fine-tuning Validation AUC (per-recording)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('logs/finetune_curves.png', dpi=120)
        plt.show()
        print(f'Best val AUC: {max(val_auc):.4f} at epoch {epochs[val_auc.index(max(val_auc))]}')
        print('Saved -> logs/finetune_curves.png')